# Unified Multimodal Training: VideoMAE + MediaPipe Hands

This notebook implements a **Unified Multimodal Classifier** combining:
1. **VideoMAE (Transformer)** to extract high-level spatial-temporal action features from video frames.
2. **MediaPipe Hands (Keypoints)** to dynamically extract coordinates of left and right hand joints in real-time on each frame.

Both feature branches are fused together and passed to a joint classification head to output the driver distraction class.

### Kaggle Setup
1. **Add-ons -> Secrets** -> add your `HF_TOKEN` (Hugging Face write token).
2. Enable **GPU** (T4 or P100) and **Internet** connection.
3. Run all cells top-to-bottom.

## 1. Install Dependencies

In [ ]:
# Install mediapipe (Tasks API requires >=0.10) and other deps
!pip install -q 'mediapipe>=0.10' transformers accelerate scikit-learn pillow opencv-python wandb tqdm
print("Dependencies installed successfully!")

In [ ]:
# Download MediaPipe Hand Landmarker model to /tmp (avoids errno=11 on Kaggle)
HAND_MODEL_PATH = '/tmp/hand_landmarker.task'
if not os.path.exists(HAND_MODEL_PATH):
    !wget -q -O $HAND_MODEL_PATH https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task
    print('Hand Landmarker model downloaded to /tmp')
else:
    print('Hand Landmarker model already present')

## 2. Environment Setup & Configuration

In [ ]:
import os
import random
import numpy as np
import torch

# Set seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Kaggle Paths
HF_CACHE_DIR = "/tmp/hf_cache"
OUTPUT_DIR = "/kaggle/working/unified_outputs"
os.makedirs(HF_CACHE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Config
MODEL_ID = "MCG-NJU/videomae-base-finetuned-kinetics"
BATCH_SIZE = 4
NUM_EPOCHS = 15
LEARNING_RATE = 1e-4
FREEZE_BACKBONE = True  # Set to False to fine-tune the VideoMAE transformer too
USE_WANDB = True

print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 3. Hugging Face & Weights & Biases Authentication

In [ ]:
from huggingface_hub import login
import os
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    login(token=token, add_to_git_credential=False)
    print("Logged in successfully via Kaggle Secrets!")
except Exception as e:
    print("Kaggle Secrets HF_TOKEN not found, please log in manually if needed.")

# Weights & Biases login
if USE_WANDB:
    try:
        from kaggle_secrets import UserSecretsClient
        wkey = UserSecretsClient().get_secret("WANDB_API_KEY")
        if wkey:
            os.environ["WANDB_API_KEY"] = wkey
            print("Loaded WANDB_API_KEY from Kaggle Secrets.")
    except Exception:
        pass
    try:
        import wandb
        wandb.login(key=os.environ.get("WANDB_API_KEY"))
        print("WandB login successful.")
    except Exception as e:
        print(f"WandB login prompt/failure: {e}")
else:
    os.environ["WANDB_MODE"] = "disabled"
    os.environ["WANDB_DISABLED"] = "true"

## 4. Download Driving Distraction Dataset

In [ ]:
# Clone the repository containing download scripts
REPO_DIR = "/kaggle/working/Driving_Distraction_Detection"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/AnnikaUnmuessig/Driving_Distraction_Detection.git {REPO_DIR}

os.chdir(REPO_DIR)
print(f"Current working directory: {os.getcwd()}")

# Download a subset of the DMD dataset
VIDEOS_PER_CLASS = 'safe_driving:15,texting_right:15,phonecall_right:15,texting_left:15,phonecall_left:15,radio:15,drinking:15,reach_side:15,hair_and_makeup:15,change_gear:15,talking_to_passenger:0'
!python scripts/download_assets.py --output_dir {HF_CACHE_DIR} --seed {SEED} --videos_per_class {VIDEOS_PER_CLASS}

## 5. Multimodal Dataset Class (MediaPipe Extractor)

In [ ]:
import cv2
from PIL import Image
import mediapipe as mp
from mediapipe.tasks import python as mp_tasks
from torch.utils.data import Dataset

class DMDMultimodalDataset(Dataset):
    def __init__(self, video_paths, labels, processor, num_frames=16):
        self.video_paths = video_paths
        self.labels = labels
        self.processor = processor
        self.num_frames = num_frames
        self.hand_landmarker = None  # lazy-init per DataLoader worker

    def __len__(self):
        return len(self.video_paths)

    def _init_landmarker(self):
        """Lazy-init HandLandmarker with CPU delegate (Tasks API)."""
        base_options = mp_tasks.BaseOptions(
            with open(HAND_MODEL_PATH, "rb") as _f:
                _model_buf = _f.read()
            base_options = mp_tasks.BaseOptions(
                model_asset_buffer=_model_buf,
                delegate=mp_tasks.BaseOptions.Delegate.CPU
            )
            delegate=mp_tasks.BaseOptions.Delegate.CPU
        )
        options = mp_tasks.vision.HandLandmarkerOptions(
            base_options=base_options,
            num_hands=2,
            min_hand_detection_confidence=0.4,
            running_mode=mp_tasks.vision.RunningMode.IMAGE
        )
        self.hand_landmarker = mp_tasks.vision.HandLandmarker.create_from_options(options)

    def _extract_landmarks(self, frame_rgb):
        if self.hand_landmarker is None:
            self._init_landmarker()
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
        result = self.hand_landmarker.detect(mp_image)
        # 126-dim vector: Left (63) + Right (63)
        left_hand_coords  = np.zeros((21, 3), dtype=np.float32)
        right_hand_coords = np.zeros((21, 3), dtype=np.float32)
        if result.hand_landmarks and result.handedness:
            for hand_landmarks, handedness in zip(result.hand_landmarks, result.handedness):
                side = handedness[0].category_name  # 'Left' or 'Right'
                coords = np.array([[lm.x, lm.y, lm.z] for lm in hand_landmarks], dtype=np.float32)
                if side == 'Left':
                    left_hand_coords = coords
                elif side == 'Right':
                    right_hand_coords = coords
        return np.concatenate([left_hand_coords.flatten(), right_hand_coords.flatten()])

    def __getitem__(self, idx):
        video_path = self.video_paths[idx]
        label = self.labels[idx]
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 1
        # Sample num_frames uniformly
        indices = set(np.linspace(0, total_frames - 1, self.num_frames, dtype=int).tolist())
        frames = []
        landmarks_seq = []
        frame_idx = 0
        success, frame = cap.read()
        while success:
            if frame_idx in indices:
                frames.append(frame)
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                landmarks_seq.append(self._extract_landmarks(frame_rgb))
            frame_idx += 1
            if len(frames) >= self.num_frames:
                break
            success, frame = cap.read()
        cap.release()
        # Zero-pad if video is shorter than expected
        while len(frames) < self.num_frames:
            frames.append(np.zeros((224, 224, 3), dtype=np.uint8))
            landmarks_seq.append(np.zeros(126, dtype=np.float32))
        # Preprocess for VideoMAE
        pil_frames = [Image.fromarray(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)) for f in frames]
        inputs = self.processor(images=pil_frames, return_tensors='pt')
        pixel_values = inputs['pixel_values'].squeeze(0)  # (16, 3, 224, 224)
        landmarks_tensor = torch.tensor(np.array(landmarks_seq), dtype=torch.float32)  # (16, 126)
        return pixel_values, landmarks_tensor, torch.tensor(label, dtype=torch.long)

## 6. Define the Custom Multimodal Classifier

In [ ]:
import torch.nn as nn
from transformers import VideoMAEModel

class UnifiedDistractionClassifier(nn.Module):
    def __init__(self, model_id, num_classes=10, cache_dir=None):
        super().__init__()
        # 1. VideoMAE Backbone (Feature Extractor)
        self.videomae = VideoMAEModel.from_pretrained(model_id, cache_dir=cache_dir)
        
        # 2. Landmark Sequence Recurrent Network (LSTM)
        # input_size = 126 (2 hands * 21 keypoints * 3 coordinates)
        self.lstm = nn.LSTM(
            input_size=126,
            hidden_size=128,
            num_layers=2,
            batch_first=True,
            bidirectional=True
        )
        
        # 3. Fusion Layer &n Classification Head
        # VideoMAEPooled: 768-d | Bi-LSTM Hidden: 256-d (128 * 2)
        self.fc = nn.Sequential(
            nn.Linear(768 + 256, 512),
            nn.LayerNorm(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )
        
    def forward(self, pixel_values, landmark_coords):
        # Forward VideoMAE
        # pixel_values shape: (batch, 16, 3, 224, 224)
        outputs = self.videomae(pixel_values)
        # Pool features over patches / tokens: (batch, 768)
        video_features = outputs.last_hidden_state.mean(dim=1)
        
        # Forward LSTM for MediaPipe landmarks
        # landmark_coords shape: (batch, 16, 126)
        lstm_out, _ = self.lstm(landmark_coords)
        # Pool features over sequence steps: (batch, 256)
        landmark_features = lstm_out.mean(dim=1)
        
        # Feature Fusionnn        fused = torch.cat([video_features, landmark_features], dim=1)  # (batch, 1024)
        
        # Classification Head
        logits = self.fc(fused)
        return logits

## 7. Prepare Dataset Splits & DataLoaders

In [ ]:
import glob
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from transformers import VideoMAEImageProcessor

# Map distraction classes to indexes
CLASSES = [
    "safe_driving", "texting_right", "phonecall_right", "texting_left", "phonecall_left",
    "radio", "drinking", "reach_side", "hair_and_makeup", "change_gear"
]
class2idx = {name: i for i, name in enumerate(CLASSES)}

# Find all downloaded video clipsnDATASET_ROOT = os.path.join(HF_CACHE_DIR, "distraction_dataset")
all_videos = []
all_labels = []

for c_name in CLASSES:
    c_dir = os.path.join(DATASET_ROOT, c_name)
    if os.path.exists(c_dir):
        v_files = glob.glob(os.path.join(c_dir, "*.mp4")) + glob.glob(os.path.join(c_dir, "*.avi"))
        for v_path in v_files:
            all_videos.append(v_path)
            all_labels.append(class2idx[c_name])

print(f"Total dataset clips found: {len(all_videos)}")

# Split train/validation (80% / 20%)
train_videos, val_videos, train_labels, val_labels = train_test_split(
    all_videos, all_labels, test_size=0.2, random_state=SEED, stratify=all_labels
)

# Initialize Image Processor
processor = VideoMAEImageProcessor.from_pretrained(MODEL_ID, cache_dir=HF_CACHE_DIR)

# Initialize Datasets
train_dataset = DMDMultimodalDataset(train_videos, train_labels, processor)
val_dataset = DMDMultimodalDataset(val_videos, val_labels, processor)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

## 8. Initialize Model, Optimizer & Loss

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = UnifiedDistractionClassifier(MODEL_ID, num_classes=len(CLASSES), cache_dir=HF_CACHE_DIR)

# Optionally freeze VideoMAE backbone
if FREEZE_BACKBONE:
    for param in model.videomae.parameters():
        param.requires_grad = False
    print("VideoMAE backbone frozen. Only training classification head & LSTM.")
else:
    print("VideoMAE backbone unfrozen. Fine-tuning entire model.")

model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE, weight_decay=0.01)

## 9. Training and Validation Loop

In [ ]:
from tqdm.notebook import tqdm

if USE_WANDB:
    import wandb
    wandb.init(
        project="driving-distraction-unified",
        name="videomae-mediapipe-multimodal",
        config={
            "model_id": MODEL_ID,
            "batch_size": BATCH_SIZE,
            "epochs": NUM_EPOCHS,
            "lr": LEARNING_RATE,
            "freeze_backbone": FREEZE_BACKBONE
        }
    )

best_val_acc = 0.0

for epoch in range(NUM_EPOCHS):
    # ── TRAINING PHASE ──
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]")
    for pixel_values, landmark_coords, labels in pbar:
        pixel_values = pixel_values.to(device)
        landmark_coords = landmark_coords.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        logits = model(pixel_values, landmark_coords)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * labels.size(0)
        _, preds = torch.max(logits, 1)
        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)
        
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})
        
    epoch_train_loss = train_loss / train_total
    epoch_train_acc = train_correct / train_total
    
    # ── VALIDATION PHASE ──
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for pixel_values, landmark_coords, labels in val_loader:
            pixel_values = pixel_values.to(device)
            landmark_coords = landmark_coords.to(device)
            labels = labels.to(device)
            
            logits = model(pixel_values, landmark_coords)
            loss = criterion(logits, labels)
            
            val_loss += loss.item() * labels.size(0)
            _, preds = torch.max(logits, 1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)
            
    epoch_val_loss = val_loss / val_total
    epoch_val_acc = val_correct / val_total
    
    print(f"Epoch {epoch+1:02d}/{NUM_EPOCHS:02d} | "
          f"Train Loss: {epoch_train_loss:.4f} - Train Acc: {epoch_train_acc*100:.1f}% | "
          f"Val Loss: {epoch_val_loss:.4f} - Val Acc: {epoch_val_acc*100:.1f}%")
          
    # Log to WandB
    if USE_WANDB:
        wandb.log({
            "epoch": epoch + 1,
            "train_loss": epoch_train_loss,
            "train_acc": epoch_train_acc,
            "val_loss": epoch_val_loss,
            "val_acc": epoch_val_acc
        })
        
    # Save intermediate checkpoint in OUTPUT_DIR (/kaggle/working/) so they are maintained
    chk_path = os.path.join(OUTPUT_DIR, f"checkpoint_epoch_{epoch+1}.pth")
    torch.save(model.state_dict(), chk_path)
    
    # Save best checkpoint
    if epoch_val_acc > best_val_acc:
        best_val_acc = epoch_val_acc
        best_path = os.path.join(OUTPUT_DIR, "best_multimodal_model.pth")
        torch.save(model.state_dict(), best_path)
        print(f"  [SAVE] New best validation accuracy: {best_val_acc*100:.1f}% - Saved best model to {best_path}")
        
if USE_WANDB:
    wandb.finish()
print(f"\nTraining complete! Best validation accuracy: {best_val_acc*100:.1f}%")